## Indicaciones sobre la Tarea Académica

> Investigaciones donde eligen su propio camino a partir de una pregunta general, documentando el proceso en un notebook interactivo.
Los asistentes de IA son herramientas de apoyo; el pensamiento crítico y la justificación de decisiones son indispensables.
La responsabilidad final de lo que presenten es suya; no poder explicar un trabajo o atribuido a la IA invalidará la tarea.


## Preguntas Guía para la Investigación

¿Cómo podemos interpretar el espacio latente de un autoencoder desde una perspectiva bayesiana vs. frecuentista? Explora las diferencias conceptuales y matemáticas.

¿Cómo podríamos modificar la función de pérdida de un autoencoder para incorporar principios bayesianos? Explora autoencoders variacionales y su relación con la inferencia bayesiana.

¿De qué manera podemos cuantificar la incertidumbre en las representaciones del espacio latente? Investiga cómo aplicar principios de estadística bayesiana para lograrlo.

Explora el concepto de "manifold learning" en autoencoders y cómo se relaciona con PCA. ¿Cuáles son las implicaciones teóricas y prácticas de estas diferencias?

Investiga cómo el espacio latente de un autoencoder puede ser regularizado para reflejar propiedades específicas de un dominio (física, química, biología), manteniendo la capacidad de reconstrucción.

# Data y definición del Modelo

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the FashionMNIST dataset
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# 2. Define the VAE
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super(VAE, self).__init__()
        # Encoder: produce features, then separate heads for mu and log_var
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, log_var = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var


# Entrenamiento

In [ ]:
latent_dim = 32
vae = VAE(latent_dim)

# ELBO loss = Reconstruction (BCE) + KL divergence
def vae_loss(recon_x, x, mu, log_var):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    # KL[ q(z|x) || N(0,I) ]
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return (bce + kl) / x.size(0)

optimizer = optim.Adam(vae.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae.to(device)

epochs = 10
for epoch in range(epochs):
    vae.train()
    running_loss = 0.0
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)

        recon, mu, log_var = vae(images)
        loss = vae_loss(recon, images, mu, log_var)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")


# Revisar resultados con el test

In [ ]:
# 5. Visualize Reconstructed Images
vae.eval()
test_images, _ = next(iter(test_loader))
test_images = test_images.view(test_images.size(0), -1).to(device)
with torch.no_grad():
    reconstructed, _, _ = vae(test_images)

test_images = test_images.view(-1, 28, 28).cpu()
reconstructed = reconstructed.view(-1, 28, 28).cpu()

n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(test_images[i], cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed[i], cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.suptitle("Original (top) vs Reconstructed (bottom)", fontsize=14)
plt.show()


# Generar nuevas imagenes desde el espacio latente

In [ ]:
# 6. Generate New Images — sample z ~ N(0, I), then decode
random_latent_vectors = torch.randn((10, latent_dim)).to(device)

with torch.no_grad():
    generated_images = vae.decoder(random_latent_vectors)

generated_images = generated_images.view(-1, 28, 28).cpu()

plt.figure(figsize=(20, 4))
for i in range(10):
    ax = plt.subplot(1, 10, i + 1)
    plt.imshow(generated_images[i], cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.suptitle("Generated Images from N(0,I) Latent Samples", fontsize=16)
plt.show()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Sliders in [-3, 3] — latent space is now regularized to N(0,1)
sliders = []
for i in range(latent_dim):
    slider = widgets.FloatSlider(
        value=0.0,
        min=-3.0,
        max=3.0,
        step=0.05,
        description=f"z{i}",
        continuous_update=False
    )
    sliders.append(slider)

test_id_slider = widgets.IntSlider(
    value=0, min=0, max=len(test_dataset)-1, step=1, description="Test ID"
)

set_from_test_button = widgets.Button(
    description="Set from Test Image", button_style="primary"
)

output_widget = widgets.Output()

def update_decoded_image(_):
    z_tensor = torch.tensor([s.value for s in sliders], device=device, dtype=torch.float32).unsqueeze(0)
    vae.eval()
    with torch.no_grad():
        generated = vae.decoder(z_tensor)
    generated_img = generated.view(28, 28).cpu().numpy()
    with output_widget:
        clear_output()
        plt.figure(figsize=(3, 3))
        plt.imshow(generated_img, cmap='gray')
        plt.title("Latent → Decoded Image")
        plt.axis('off')
        plt.show()

def set_sliders_from_test_image(_):
    idx = test_id_slider.value
    test_image, _ = test_dataset[idx]
    test_image_flat = test_image.view(1, -1).to(device)
    vae.eval()
    with torch.no_grad():
        h = vae.encoder(test_image_flat)
        mu = vae.fc_mu(h)                  # use mean (no noise) for the widget
    encoded_values = mu[0].cpu().numpy()
    for i, val in enumerate(encoded_values):
        sliders[i].value = float(np.clip(val, -3.0, 3.0))

for slider in sliders:
    slider.observe(update_decoded_image, names='value')
set_from_test_button.on_click(set_sliders_from_test_image)

controls_box = widgets.VBox([
    widgets.HBox([test_id_slider, set_from_test_button]),
    widgets.VBox(sliders)
])
ui = widgets.HBox([controls_box, output_widget])
display(ui)
update_decoded_image(None)
